# Urdu Orpheus on Kaggle — complete fresh-runtime setup
Import **this updated notebook** into Kaggle; updating the cloned repository does not update existing notebook cells.
Enable GPU and Internet, add the HF_TOKEN Secret, and edit HF_REPO_ID and SESSION_HOURS in Cell 1. The repository URL is already filled in.
Cell 2 always installs dependencies before checking imports. Do not replace it with an import-only cell.
Use Save Version → Save & Run All for unattended execution. Run a separate smoke experiment before the full epoch; see README.md.
Default policy skips small bad/duplicate/context-overflow subsets and logs every exclusion. Training retention limits are in configs/aslp50h.json.
Recovery now queues failed uploads locally, regenerates damaged encoding caches and probes GPU backward memory automatically. See REVIEW.md for tested coverage and remaining limits.


Cell 1 — clone a fixed version

In [ ]:
import os, sys, subprocess, re
from pathlib import Path

REPO_URL = "https://github.com/hassan-31x/orpheus-ft-urdu.git"
REPO_REF = "main"  # Use the same fixed commit for an experiment and its resumes.
HF_REPO_ID = "hassan-31x/orpheus-urdu-checkpoints"  # Your private model repository.
AUDIT_ONLY = False  # Set True with GPU OFF to inspect data before training.
MINIMUM_TRAIN_HOURS = 35  # The 47h archive contains ~38h train plus validation/test.
SESSION_HOURS = 10.5  # Set below the limit displayed in your Kaggle account.
REPO = Path("/kaggle/working/urdu-orpheus")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", HF_REPO_ID):
    raise ValueError("HF_REPO_ID must have the format username/repository")
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Add the HF_TOKEN Kaggle Secret and enable access for this notebook") from None
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
# Fetch even when the directory already exists; checkout alone leaves old code.
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
for name in ("setup_kaggle.py", "check_environment.py", "requirements-kaggle.txt", "pipeline_recovery.py"):
    if not (REPO / name).is_file():
        raise RuntimeError(f"Repository version lacks {name}. Upload the updated project before running.")
print("Project commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


Cell 2 — install and preflight

In [ ]:
# Always install in this runtime: saved Kaggle jobs start with a fresh environment.
ENV = Path("/kaggle/working/orpheus-env")
TRAIN_PYTHON = str(ENV / "bin" / "python")
setup_command = [sys.executable, "-u", str(REPO / "setup_kaggle.py"), "--venv", str(ENV)]
if not AUDIT_ONLY:
    setup_command.append("--gpu")
subprocess.run(setup_command, cwd=REPO, check=True)
assert Path(TRAIN_PYTHON).is_file(), "Training environment was not created"
print("All training subprocesses will use:", TRAIN_PYTHON)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)


Cell 3 — private credentials and config

In [ ]:
import json
cfg = json.loads((REPO / "configs/aslp50h.json").read_text())
cfg["checkpoint_backend"] = "huggingface"
cfg["hf_repo_id"] = HF_REPO_ID
cfg["session_hours"] = SESSION_HOURS
cfg["minimum_train_hours"] = MINIMUM_TRAIN_HOURS
print("Minimum retained TRAIN split hours:", cfg["minimum_train_hours"])
print("Retention thresholds:", "enforced" if cfg["enforce_retention_limits"] else "warnings only")
print("Data policy:", cfg["invalid_row_policy"], "maximum training exclusion fraction:", cfg["maximum_rejected_train_fraction"])
CONFIG = Path("/kaggle/working/run-config.json")
CONFIG.write_text(json.dumps(cfg, indent=2))
RUN = Path(cfg["work_dir"]) / "runs" / cfg["run_id"]
subprocess.run([TRAIN_PYTHON, str(REPO / "finetune_aslp_50h.py"),
                "--config", str(CONFIG), "--mode", "storage-check"], cwd=REPO, check=True)


Cell 4A — recommended unattended saved run

In [ ]:
# Blocks until the audit/training process completes. Failures print diagnostics.
command = [TRAIN_PYTHON, "-u", str(REPO / "finetune_aslp_50h.py"), "--config", str(CONFIG)]
if AUDIT_ONLY:
    command += ["--mode", "audit"]
try:
    subprocess.run(command, cwd=REPO, check=True)
except subprocess.CalledProcessError:
    for name in ("data_error_summary.json", "data_errors.json"):
        path = RUN / name
        if path.exists():
            value = json.loads(path.read_text())
            print("Diagnostic file:", path)
            print(json.dumps(value if isinstance(value, dict) else value[:10], indent=2, ensure_ascii=False))
    print("The subprocess stopped. Training does not restart automatically; inspect the error before rerunning.")
    raise


Cell 5 — status and latest samples

In [ ]:
from IPython.display import Audio, display
status = RUN / "status.json"
print(json.loads(status.read_text()) if status.exists() else ("Dataset audit only; no training started" if AUDIT_ONLY else "No training status recorded"))
for path in sorted((RUN / "samples").glob("step-*/*.wav"))[-3:]:
    print(path)
    try:
        display(Audio(filename=str(path)))
    except (OSError, ValueError) as exc:
        print("Optional playback unavailable:", type(exc).__name__)

backup = RUN / "backup_status.json"
if backup.exists():
    print("Remote backup:", json.loads(backup.read_text()))


Cell 6 — test the final adapter after completion

In [ ]:
if AUDIT_ONLY:
    print("Dataset audit completed; set AUDIT_ONLY=False and enable GPU for training.")
else:
    status_value = json.loads((RUN / "status.json").read_text())
    if status_value["status"] in ("complete", "smoke_complete"):
        output = Path("/kaggle/working/urdu_test.wav")
        try:
            subprocess.run([TRAIN_PYTHON, str(REPO / "synthesize.py"),
                            "--adapter", str(RUN / "adapter_final"),
                            "--text", "آج موسم بہت خوشگوار ہے اور ہم سب باہر سیر کے لیے جا رہے ہیں۔",
                            "--output", str(output)], cwd=REPO, check=True)
            display(Audio(filename=str(output)))
        except subprocess.CalledProcessError as exc:
            print("Training adapter is saved at", RUN / "adapter_final")
            print("Optional final synthesis failed; training remains complete. See synthesis_status.json.")
            (RUN / "synthesis_status.json").write_text(json.dumps({"status": "failed", "returncode": exc.returncode}))

    else:
        print("Saved for continuation; rerun the same config to finish the epoch.")
